# 01 · Lakebase project `bluerridge-pm`, five synced tables, and the "why Lakebase" check

The app's hot path (KPI strip, compliance list, risk card, "already late" list, seeded daily brief) reads from Lakebase Postgres. This notebook:

1. Confirms the Lakebase project `bluerridge-pm`, its `production` branch and the `primary` read-write endpoint are `ACTIVE` at 1 CU (waits if not).
2. Confirms the five synced tables exist in `mfg_mc_se_sa.bluerridge_app` and are online with a finished sync.
3. Connects to Postgres with **psycopg** using a database credential generated for the identity running this notebook, and asserts that every Postgres row count equals the Unity Catalog source count.
4. Shows three sample rows from `kpi_daily_synced` and `risk_forecast_synced`.
5. Compares latency of the KPI-strip query (the as-of `kpi_daily` row) on Lakebase against the serverless SQL warehouse `hobbs`: 20 calls each, p50 and p95, plus the first (cold) call reported separately.
6. Writes `outputs/synced_tables_listing.md` from the values it just read.

**Bundle or SDK?** The project and the five synced tables are bundle resources (`resources/lakebase.yml`: `postgres_projects` and `postgres_synced_tables`, both in the CLI v1.20.0 bundle schema), created by `databricks bundle deploy -t demo`. This notebook only *confirms* state. The retired Provisioned tier (`database_instances`) is not used: Databricks Apps bind Lakebase through the `postgres` resource key (branch + database), which targets Autoscaling projects. See `docs/BUILD.md`, "Lakebase + app".

**Synced tables are SNAPSHOT mode.** The gold sources are rewritten with `CREATE OR REPLACE` and carry no Change Data Feed, so TRIGGERED/CONTINUOUS would not work. A refresh is a re-sync after each gold refresh.

**`drawing_register` is deliberately not synced.** It carries a Unity Catalog row filter; a Postgres copy would not enforce it. The app reads it from the SQL warehouse, where the filter applies.

**No AI calls and no Spark SQL.** The notebook uses the Databricks SDK (Postgres API, Statement Execution API) and psycopg only, so there is no query plan to check for AI steps; the plan check is skipped on purpose. Cost: one small serverless run of a few minutes against an already-running project.

**Latency method.** Both sides run the same logical query, the newest `kpi_daily` row for project `BRL-PH2` with `snapshot_date <=` an as-of date, with 20 *different* as-of dates so neither side can answer from a result cache. Lakebase is timed on one open connection (what the app's pool does), the warehouse through the Statement Execution API (HTTP round trip included, which is how the app would call it). Timings are wall-clock from this serverless notebook, so they include network. The warehouse is also timed with one identical statement repeated 20 times, which is eligible for the result cache, so you can see the fast case honestly.

In [1]:
%pip install --quiet --upgrade databricks-sdk "psycopg[binary]>=3.1"
dbutils.library.restartPython()

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googleapis-common-protos 1.62.0 requires protobuf!=3.20.0,!=3.20.1,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0.dev0,>=3.19.5, but you have protobuf 7.36.2 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [2]:
import json, os, statistics, time
import psycopg
from databricks.sdk import WorkspaceClient

dbutils.widgets.text("run_id", "interactive")
RUN_ID = dbutils.widgets.get("run_id")

CATALOG, SRC_SCHEMA, APP_SCHEMA = "mfg_mc_se_sa", "bluerridge_gold_pm", "bluerridge_app"
PROJECT = "projects/bluerridge-pm"
BRANCH = f"{PROJECT}/branches/production"
ENDPOINT = f"{BRANCH}/endpoints/primary"
PG_DB = "databricks_postgres"
WAREHOUSE_ID = "527214d7319cb124"
TABLES = {  # source table -> primary key
    "kpi_daily": "project_id, snapshot_date",
    "compliance_findings": "drawing_id, finding_id",
    "risk_forecast": "milestone_id",
    "schedule_overdue": "milestone_id",
    "daily_brief": "project_id, brief_date",
}
w = WorkspaceClient()
ME = w.current_user.me().user_name
print("run_id:", RUN_ID, "| running identity:", ME)

run_id: 235776026823899 | running identity: joseph.hobbs@databricks.com


## 1. Project, branch and endpoint state

In [3]:
def wait(fn, ok, what, tries=40, sleep=30):
    for i in range(tries):
        v = fn()
        if ok(v):
            return v
        print(f"  waiting for {what}: {v}")
        time.sleep(sleep)
    raise RuntimeError(f"{what} not ready")

def fmt_dur(x):
    return f"{x.seconds}s" if hasattr(x, "seconds") else str(x)

project = w.postgres.get_project(name=PROJECT)
endpoint = wait(lambda: w.postgres.get_endpoint(name=ENDPOINT),
                lambda e: str(e.status.current_state).endswith("ACTIVE"), "endpoint ACTIVE")
HOST = endpoint.status.hosts.host
POOLER_HOST = endpoint.status.hosts.read_write_pooled_host
st = endpoint.status
print("project:", project.name, "| display_name:", project.status.display_name, "| pg_version:", project.status.pg_version, "| owner:", project.status.owner)
print("endpoint:", endpoint.name, "| type:", st.endpoint_type, "| state:", st.current_state)
print("capacity (CU min/max):", st.autoscaling_limit_min_cu, "/", st.autoscaling_limit_max_cu, "| suspend_timeout:", st.suspend_timeout_duration)
print("read_write_dns:", HOST)
print("pooled host:", POOLER_HOST)
assert st.autoscaling_limit_min_cu == 1 and st.autoscaling_limit_max_cu == 1

project: projects/bluerridge-pm | display_name: Blue Ridge PM Co-Pilot hot path | pg_version: 17 | owner: joseph.hobbs@databricks.com
endpoint: projects/bluerridge-pm/branches/production/endpoints/primary | type: EndpointType.ENDPOINT_TYPE_READ_WRITE | state: EndpointStatusState.ACTIVE
capacity (CU min/max): 1.0 / 1.0 | suspend_timeout: seconds: 86400

read_write_dns: ep-silent-lake-d2qn95e6.database.us-east-1.cloud.databricks.com
pooled host: ep-silent-lake-d2qn95e6-pooler.database.us-east-1.cloud.databricks.com


## 2. Synced tables: online, with a finished sync

In [4]:
synced = {}
for t in TABLES:
    sid = f"{CATALOG}.{APP_SCHEMA}.{t}_synced"
    s = wait(lambda: w.postgres.get_synced_table(name=f"synced_tables/{sid}"),
             lambda s: str(s.status.detailed_state).split(".")[-1] == "SYNCED_TABLE_ONLINE_NO_PENDING_UPDATE", f"{sid} online")
    synced[t] = s
    ls = s.status.last_sync
    print(f"{sid}\n   state={str(s.status.detailed_state).split('.')[-1]}  last_sync_end={ls.sync_end_time.seconds if hasattr(ls.sync_end_time, 'seconds') else ls.sync_end_time} (epoch s)  uc_state={s.status.unity_catalog_provisioning_state}  pipeline={s.status.pipeline_id}")

mfg_mc_se_sa.bluerridge_app.kpi_daily_synced
   state=SYNCED_TABLE_ONLINE_NO_PENDING_UPDATE  last_sync_end=1794162097 (epoch s)  uc_state=ProvisioningInfoState.ACTIVE  pipeline=128e16b0-9320-4850-97d9-b09bd08445d5
mfg_mc_se_sa.bluerridge_app.compliance_findings_synced
   state=SYNCED_TABLE_ONLINE_NO_PENDING_UPDATE  last_sync_end=1794162102 (epoch s)  uc_state=ProvisioningInfoState.ACTIVE  pipeline=862e5ced-5e1e-4030-b3da-25161b77a8e7
mfg_mc_se_sa.bluerridge_app.risk_forecast_synced
   state=SYNCED_TABLE_ONLINE_NO_PENDING_UPDATE  last_sync_end=1794162102 (epoch s)  uc_state=ProvisioningInfoState.ACTIVE  pipeline=80ab9ecc-e330-42d2-9cdd-86e58fd4206b
mfg_mc_se_sa.bluerridge_app.schedule_overdue_synced
   state=SYNCED_TABLE_ONLINE_NO_PENDING_UPDATE  last_sync_end=1794162100 (epoch s)  uc_state=ProvisioningInfoState.ACTIVE  pipeline=2b0496be-05bc-44ad-8f17-89ee5eb7e194
mfg_mc_se_sa.bluerridge_app.daily_brief_synced
   state=SYNCED_TABLE_ONLINE_NO_PENDING_UPDATE  last_sync_end=1794162100 (ep

## 3. Postgres connection with psycopg, and row counts: Postgres vs the Unity Catalog source

In [5]:
def new_conn():
    cred = w.postgres.generate_database_credential(endpoint=ENDPOINT)   # short-lived OAuth token, never printed or stored
    return psycopg.connect(host=HOST, port=5432, dbname=PG_DB, user=ME, password=cred.token, sslmode="require", connect_timeout=30)

t0 = time.perf_counter()
conn = new_conn()
conn_ms = (time.perf_counter() - t0) * 1000
with conn.cursor() as cur:
    cur.execute("select version(), current_user, current_database()")
    print(cur.fetchone())
print(f"credential + TCP/TLS connect + first query (cold, includes any compute wake-up): {conn_ms:.0f} ms")

('PostgreSQL 17.11 (9045acd) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit', 'joseph.hobbs@databricks.com', 'databricks_postgres')
credential + TCP/TLS connect + first query (cold, includes any compute wake-up): 115 ms


In [6]:
def wh(sql):
    r = w.statement_execution.execute_statement(warehouse_id=WAREHOUSE_ID, statement=sql, wait_timeout="50s")
    assert str(r.status.state).endswith("SUCCEEDED"), r.status
    return r.result.data_array if r.result else None

counts = {}
for t in TABLES:
    uc = int(wh(f"select count(*) from {CATALOG}.{SRC_SCHEMA}.{t}")[0][0])
    with conn.cursor() as cur:
        cur.execute(f'select count(*) from {APP_SCHEMA}."{t}_synced"')
        pg = cur.fetchone()[0]
    counts[t] = (uc, pg)
    print(f"{t:22s} UC source={uc:5d}   Postgres {APP_SCHEMA}.{t}_synced={pg:5d}   {'MATCH' if uc == pg else 'MISMATCH'}")
assert all(uc == pg for uc, pg in counts.values()), counts
assert counts["kpi_daily"][1] > 0
print("all five Postgres counts equal the Unity Catalog source counts")

kpi_daily              UC source=   90   Postgres bluerridge_app.kpi_daily_synced=   90   MATCH
compliance_findings    UC source= 1379   Postgres bluerridge_app.compliance_findings_synced= 1379   MATCH
risk_forecast          UC source=   64   Postgres bluerridge_app.risk_forecast_synced=   64   MATCH
schedule_overdue       UC source=    7   Postgres bluerridge_app.schedule_overdue_synced=    7   MATCH
daily_brief            UC source=    1   Postgres bluerridge_app.daily_brief_synced=    1   MATCH
all five Postgres counts equal the Unity Catalog source counts


## 4. Sample rows read over Postgres

In [7]:
def show(table, order, n=3):
    with conn.cursor() as cur:
        cur.execute(f'select * from {APP_SCHEMA}."{table}" order by {order} limit {n}')
        cols = [d.name for d in cur.description]
        rows = cur.fetchall()
    print(f"-- {APP_SCHEMA}.{table}  ({n} rows)")
    for r in rows:
        print(json.dumps(dict(zip(cols, r)), default=str))
    return cols, rows

kpi_cols, kpi_rows = show("kpi_daily_synced", "snapshot_date desc")
risk_cols, risk_rows = show("risk_forecast_synced", "risk_rank")

-- bluerridge_app.kpi_daily_synced  (3 rows)
{"project_id": "BRL-PH2", "snapshot_date": "2026-11-06", "is_current": true, "schedule_status": "on_track", "milestones_hit": 6, "milestones_total": 11, "slipped_milestone_id": "MS-281", "slipped_days": 4, "budget_total_usd": 208700000.0, "budget_overrun_usd": 4800000.0, "budget_eac_pct": 2.3, "safety_days_without_incident": 28, "recordables_28d": 0, "trir_28d": 0.0, "open_ncrs": 24, "open_ncrs_compliance": 9, "open_ncrs_workmanship": 15, "compliance_reviewed_week": 238, "compliance_passed_week": 187, "compliance_failed_week": 51, "compliance_review_week": 0, "computed_at": "2026-11-08 16:29:43.508835+00:00"}
{"project_id": "BRL-PH2", "snapshot_date": "2026-11-05", "is_current": false, "schedule_status": "on_track", "milestones_hit": 6, "milestones_total": 11, "slipped_milestone_id": "MS-281", "slipped_days": 3, "budget_total_usd": 208700000.0, "budget_overrun_usd": 4800000.0, "budget_eac_pct": 2.3, "safety_days_without_incident": 27, "recor

## 5. Latency: KPI-strip query on Lakebase vs the SQL warehouse

The query is "the newest `kpi_daily` row for `BRL-PH2` at or before an as-of date". The first call on each side is reported separately (it pays connection or warehouse-routing costs); the 20 timed calls follow, each with a different as-of date.

In [8]:
import datetime as dt
PROJECT_ID = "BRL-PH2"
ASOF = [dt.date(2026, 11, 6) - dt.timedelta(days=i) for i in range(21)]   # 21 distinct dates
COLD_DATE, DATES = ASOF[0], ASOF[1:]                                       # first call separate, then 20 timed

def pg_kpi(asof):
    t = time.perf_counter()
    with conn.cursor() as cur:
        cur.execute(f'select * from {APP_SCHEMA}.kpi_daily_synced where project_id = %s and snapshot_date <= %s order by snapshot_date desc limit 1', (PROJECT_ID, asof))
        row = cur.fetchone()
    ms = (time.perf_counter() - t) * 1000
    assert row is not None
    return ms

def wh_kpi(asof):
    t = time.perf_counter()
    rows = wh(f"select * from {CATALOG}.{SRC_SCHEMA}.kpi_daily where project_id = '{PROJECT_ID}' and snapshot_date <= DATE '{asof}' order by snapshot_date desc limit 1")
    ms = (time.perf_counter() - t) * 1000
    assert rows and len(rows) == 1
    return ms

def pct(xs, p):
    xs = sorted(xs); k = (len(xs) - 1) * p / 100; f = int(k); c = min(f + 1, len(xs) - 1)
    return xs[f] + (xs[c] - xs[f]) * (k - f)

# a fresh connection for the Lakebase timing: first call = first query on a new connection (credential generation timed separately above)
conn.close(); conn = new_conn()
lb_first = pg_kpi(COLD_DATE)
lb = [pg_kpi(d) for d in DATES]
wh_first = wh_kpi(COLD_DATE)
whs = [wh_kpi(d) for d in DATES]
SAME = DATES[0]
wh_same_first = wh_kpi(SAME)
wh_same = [wh_kpi(SAME) for _ in range(20)]

rows = [("Lakebase (psycopg, open connection)", lb_first, lb),
        ("SQL warehouse hobbs (Statement API, distinct as-of dates)", wh_first, whs),
        ("SQL warehouse hobbs (identical statement, result-cache eligible)", wh_same_first, wh_same)]
print(f"{'path':68s} {'first ms':>9s} {'p50 ms':>8s} {'p95 ms':>8s} {'min':>7s} {'max':>8s}  n")
for name, first, xs in rows:
    print(f"{name:68s} {first:9.1f} {pct(xs,50):8.1f} {pct(xs,95):8.1f} {min(xs):7.1f} {max(xs):8.1f}  {len(xs)}")
print("Lakebase connect (credential + TLS + first query on this run):", round(conn_ms), "ms")
print("p50 ratio warehouse(distinct) / Lakebase = %.1fx" % (pct(whs, 50) / pct(lb, 50)))

path                                                                  first ms   p50 ms   p95 ms     min      max  n
Lakebase (psycopg, open connection)                                        6.8      3.0      5.8     2.8      7.2  20
SQL warehouse hobbs (Statement API, distinct as-of dates)                432.5    428.2    471.1   376.3    479.3  20
SQL warehouse hobbs (identical statement, result-cache eligible)         480.4    426.9    488.1   377.9    492.1  20
Lakebase connect (credential + TLS + first query on this run): 115 ms
p50 ratio warehouse(distinct) / Lakebase = 143.3x


## 6. Write the evidence listing

In [9]:
import datetime as _dt
def fmt(x):
    # the Postgres API returns protobuf Timestamp / Duration objects; render them as plain text
    if hasattr(x, "nanos") and hasattr(x, "seconds") and x.seconds > 10**8:
        return _dt.datetime.fromtimestamp(x.seconds, _dt.timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    if hasattr(x, "seconds"):
        return f"{x.seconds}s"
    return str(x)

os.makedirs("outputs", exist_ok=True)
ep_st = endpoint.status
lines = []
A = lines.append
A("# Lakebase synced tables listing")
A("")
A(f"Generated by `08_lakebase/01_create_instance.ipynb` (serverless run `{RUN_ID}`) from values read at run time. No credentials are stored here.")
A("")
A("## Lakebase project")
A("")
A("| Field | Value |")
A("|---|---|")
A(f"| Flavor | Lakebase Autoscaling Postgres project (bundle resource `postgres_projects.bluerridge_pm`) |")
A(f"| Project (the \"instance\") | `bluerridge-pm` (`{project.name}`), Postgres {project.status.pg_version}, owner {project.status.owner} |")
A(f"| Branch | `production` |")
A(f"| Endpoint | `primary`, {str(ep_st.endpoint_type).split('.')[-1]}, state {str(ep_st.current_state).split('.')[-1]} |")
A(f"| Capacity | {ep_st.autoscaling_limit_min_cu:g} CU min / {ep_st.autoscaling_limit_max_cu:g} CU max (fixed 1 CU; scale-to-zero suspend timeout {fmt(ep_st.suspend_timeout_duration)}) |")
A(f"| `read_write_dns` | `{HOST}` |")
A(f"| Pooled host | `{POOLER_HOST}` |")
A(f"| Postgres database | `{PG_DB}` (resource `{BRANCH}/databases/databricks-postgres`) |")
A(f"| Postgres schema | `{APP_SCHEMA}` |")
A("")
A("## Synced tables")
A("")
A("All five are SNAPSHOT-scheduled (the gold sources are rewritten with `CREATE OR REPLACE` and have no Change Data Feed). Refresh means re-syncing after a gold refresh.")
A("")
A("| Source (Unity Catalog) | Synced table (Unity Catalog) | Postgres table the app queries | Primary key | Mode | State | Last sync end | Rows UC | Rows PG | Match |")
A("|---|---|---|---|---|---|---|---|---|---|")
for t, pk in TABLES.items():
    s = synced[t]; uc, pg = counts[t]
    A(f"| `{CATALOG}.{SRC_SCHEMA}.{t}` | `{CATALOG}.{APP_SCHEMA}.{t}_synced` | `{APP_SCHEMA}.{t}_synced` | {pk} | SNAPSHOT | {str(s.status.detailed_state).split('.')[-1]} | {fmt(s.status.last_sync.sync_end_time)} | {uc} | {pg} | {'yes' if uc == pg else 'NO'} |")
A("")
A("## Latency: KPI-strip query, Lakebase vs SQL warehouse")
A("")
A(f"Query: newest `kpi_daily` row for `{PROJECT_ID}` at or before an as-of date. Wall-clock from a serverless notebook, 20 timed calls per row (different as-of dates, so no result-cache hits, except the last row). 'First' is the first call, reported separately. Lakebase connect (credential + TLS + first query) took {conn_ms:.0f} ms.")
A("")
A("| Path | First call (ms) | p50 (ms) | p95 (ms) | min (ms) | max (ms) | n |")
A("|---|---|---|---|---|---|---|")
for name, first, xs in rows:
    A(f"| {name} | {first:.1f} | {pct(xs,50):.1f} | {pct(xs,95):.1f} | {min(xs):.1f} | {max(xs):.1f} | {len(xs)} |")
A("")
A(f"Warehouse p50 is {pct(whs,50)/pct(lb,50):.1f}x Lakebase p50 on distinct statements. The warehouse numbers include the REST round trip the app would also pay; a warm serverless warehouse is fast, and the identical-statement row shows what the result cache gives it. Lakebase is the point-read path (single indexed row, open connection); the warehouse stays the right engine for aggregations and scans.")
A("")
A("## Governance note: `drawing_register` is not synced")
A("")
A("`mfg_mc_se_sa.bluerridge_gold_pm.drawing_register` is deliberately **not** synced to Lakebase. It carries a Unity Catalog row filter (by discipline, via `bluerridge_agents.discipline_access`). Row filters and column masks are not propagated to synced tables, so a Postgres copy would hand every connected role all rows and bypass the filter. The app reads `drawing_register` through the SQL warehouse, where the filter is enforced.")
A("")
A("## Sample rows over Postgres")
A("")
for title, cols, rws in (("kpi_daily_synced", kpi_cols, kpi_rows), ("risk_forecast_synced", risk_cols, risk_rows)):
    A(f"`{APP_SCHEMA}.{title}`, 3 rows:")
    A("")
    A("| " + " | ".join(cols) + " |")
    A("|" + "---|" * len(cols))
    for r in rws:
        A("| " + " | ".join(str(v) for v in r) + " |")
    A("")
open("outputs/synced_tables_listing.md", "w").write("\n".join(lines))
open("outputs/latency_raw.json", "w").write(json.dumps({"run_id": RUN_ID, "connect_ms": conn_ms,
    "lakebase_first_ms": lb_first, "lakebase_ms": lb, "warehouse_first_ms": wh_first, "warehouse_ms": whs,
    "warehouse_same_first_ms": wh_same_first, "warehouse_same_ms": wh_same}, indent=1))
conn.close()
print("wrote outputs/synced_tables_listing.md and outputs/latency_raw.json")

wrote outputs/synced_tables_listing.md and outputs/latency_raw.json
